# Mnist Qiskit: Amplitude encoding with 10 qubits

## Imports

In [1]:
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.circuit import ParameterVector
from sklearn.datasets import fetch_openml

## Dataset and Circuits/Labels

In [2]:
mnist = fetch_openml('mnist_784', version=1)
X, y = mnist.data, mnist.target
y = mnist.target.astype(int)
circuits = []
labels = []

## Amplitude Encoding
Amplitude encoding maps a normalized classical vector $x \in \mathbb{R}^N$ into the amplitudes of an $n$-qubit quantum state $|\psi\rangle \in \mathbb{C}^{2^n}$, where $n = \lceil \log_2 N \rceil$. This approach is often considered more "quantum-native"; however, it is expensive to implement on current hardware and is generally not ideal for datasets such as MNIST.  
To apply amplitude encoding, the classical vector must satisfy:  
* Normalization: $ x \leftarrow \frac{x}{\|x\|} $
* Padding: If $x \notin \mathbb{R}^{2^n}$, it must be padded with zeros until $x \in \mathbb{R}^{2^n}$.

In [3]:
def amplitude_encode(image):
    x = np.asarray(image, dtype=float).flatten()

    norm = np.linalg.norm(x)
    if norm == 0:
        raise ValueError("Zero vector")

    x = x / norm

    padded = np.zeros(1024)
    padded[:len(x)] = x

    qc = QuantumCircuit(10)
    qc.initialize(padded, range(10))
    return qc

## Variational Quantum Model
The function bellow constructs a parameterized quantum circuit by adding layers of single-qubit rotations and entangling gates.

The model consists of repeated layers, each containing:  
* Parameterized Rotations: A rotation around the $y$-axis is applied to each qubit using $R_y(\theta)$ gates, where the parameters are drawn from a vector $\theta$.  
More precisely, the rotation on the $y$-axis is:
$$
R_y(\theta) = \begin{pmatrix}\cos{\frac{\theta}{2}} & -\sin{\frac{\theta}{2}} \\ \sin{\frac{\theta}{2}} & \cos{\frac{\theta}{2}} \end{pmatrix}
$$

* Entanglement: Controlled-NOT (CNOT) gates are applied between neighboring qubits to introduce entanglement.

More precisely the:  
* If the control qubit is in state $|0\rangle$, the target qubit remains unchanged.  
* If the control qubit is in state $|1\rangle$, the target qubit is flipped (i.e., $|0\rangle \leftrightarrow |1\rangle$).

This pattern is repeated for n_layers, resulting in a circuit with $10 \times n_{\text{layers}}$ trainable parameters.

The function returns the updated quantum circuit along with the parameter vector $\theta$.

In [4]:
def build_model(qc, n_layers=1):
    params = ParameterVector("θ", length=10 * n_layers)

    idx = 0
    for _ in range(n_layers):
        for q in range(10):
            qc.ry(params[idx], q)
            idx += 1

        for q in range(9):
            qc.cx(q, q + 1)

    return qc, params

## Forward Pass

The function bellow evaluates the quantum circuit and produces a scalar prediction from the resulting quantum state.

The procedure is:  
* Parameter binding: The variational parameters $θ$ are assigned to the circuit, producing a fully specified quantum circuit.  
* Statevector simulation: The quantum state $|ψ⟩$ is obtained by simulating the circuit.  
* Probability extraction: The probabilities of all computational basis states are computed as $p_i = |⟨i | ψ⟩|²$.  
* Measurement mapping: The probabilities are mapped to a scalar value in [-1, 1] using a parity-based weighting:
  $$
  \text{pred} = \sum_{i=0}^{2^n - 1} (-1)^{w(i)} p_i
  $$
  where $w(i)$ is the number of ones in the binary representation of $i$.

This mapping assigns a positive or negative contribution to each basis state depending on its parity, effectively computing a weighted expectation value.

The function returns the real part of the prediction.

In [5]:
def forward_pass(qc, param_values):
    bound = qc.assign_parameters(param_values, inplace=False)
    state = Statevector.from_instruction(bound).data
    probs = np.abs(state) ** 2
    pred = 0
    for i, p in enumerate(probs):
        n_ones = bin(i).count("1")
        pred += ((-1) ** n_ones) * p
    return np.real(pred)

## Loss Function

The function loss_fn(params) computes the mean squared error over a batch of quantum circuit samples.

The procedure is as follows:  
* For each quantum circuit qc in the dataset, the model prediction is obtained using the forward pass with the current parameters.  
* The squared difference between the prediction and the corresponding label is computed.  
* These errors are accumulated over the entire batch and averaged.

The resulting loss is defined as:
$$
\mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} ( \hat{y}_i - y_i )^2
$$

where $\hat{y}_i$ is the model prediction, $y_i$ is the true label, and $N$ is the number of circuits in the batch.

In [6]:
def loss_fn(params):
    total = 0.0
    for qc, label in zip(circuits, labels):
        pred = forward_pass(qc, params)
        total += (pred - label) ** 2
    return total / len(circuits)

## Training Procedure

The training process begins by setting a fixed random seed to ensure reproducibility.

The dataset is converted into numerical arrays:
* $X \in \mathbb{R}^N$ is cast to floating-point values  
* $y \in \{0,1\}$ is cast to integer labels and then mapped to $\{-1, +1\}$ using a bipolar encoding

Each sample is then converted into a quantum circuit:
* For each of the first 10 samples, amplitude encoding is applied to the input vector
* A variational quantum model with 1 layer is attached to the encoded circuit
* The resulting circuits and labels are stored for training

The model parameters are initialized uniformly in the interval $[0, 2\pi]$ for all $10$ trainable parameters.

Optimization is performed using the COBYLA method from SciPy, minimizing the defined loss function.

The final trained parameters and loss value are then extracted and displayed.

In [7]:
np.random.seed(11051918)

X = np.asarray(X, dtype=float)
y = np.asarray(y, dtype=int)

y = np.where(y == 0, -1, 1)

for i in range(10):
    qc = amplitude_encode(X[i])
    qc, _ = build_model(qc, n_layers=1)
    circuits.append(qc)
    labels.append(y[i])

n_params = 10 * 1
init_params = np.random.uniform(0, 2 * np.pi, n_params)

result = minimize(
    loss_fn,
    init_params,
    method="COBYLA"
)

trained_params = result.x

print("Trained params:", trained_params)
print("Final loss:", result.fun)

Trained params: [0.89572931 1.66777916 3.69177865 5.03215553 2.8594625  4.85001779
 4.16023451 4.08467863 5.72829141 2.64915715]
Final loss: 0.8079437858170465


## Evaluation

In [8]:
preds = []

for qc in circuits:
    pred = forward_pass(qc, trained_params)
    preds.append(pred)

preds = np.array(preds)

pred_labels = np.where(preds > 0, 1, -1)

accuracy = np.mean(pred_labels == labels)

print("Accuracy:", accuracy)

Accuracy: 0.9


## Next steps
This aproach is exploratory. I have seen these two papers on the topic but before reading those I decided to try "my own aproach". I also want try a small neural network agains this aproach just for fun, if one day this comment disappear, I implemented this first goal.  

Variational quantum models and data encoding (arXiv:2103.06232)  
Data re-uploading for universal quantum classifiers (arXiv:2008.08605)  